In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

# LLM Inference Workshop – Part 4
## Supervised Fine-Tuning an Instruct Model with `Trainer`

In this notebook we will:

1. Load GSM8K.
2. Format each example for an instruct model.
3. Build `input_ids` and `labels` for supervised fine-tuning.
4. Mask the prompt tokens with `-100`.
5. Fine-tune `meta-llama/Llama-3.2-1B-Instruct` with HuggingFace `Trainer`.
6. Compare evaluation before and after fine-tuning.

The key idea in this notebook is the training objective.

For an instruct model, the input prompt is the **context**.
The assistant response is the **target**.

Therefore:
- prompt tokens should be visible to the model
- but they should **not** contribute to the loss
- only the assistant completion should be predicted


## Practical Note

This notebook performs **full supervised fine-tuning** of a small model.

That is intentional:
- Notebook 4 focuses on the core SFT pipeline with `Trainer`
- Notebook 5 will reduce memory cost with PEFT / LoRA

If your GPU memory is limited, reduce:
- `TRAIN_SAMPLES`
- `MAX_LENGTH`
- `per_device_train_batch_size`
- `gradient_accumulation_steps`


In [2]:
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from IPython.display import Markdown

from utils import (
    extract_last_after_hashes,
    MATH_PROMPT,
    evaluate_model,
    compute_accuracy,
    replace_hash_with_boxed,
)
from vllm import LLM, SamplingParams, TokensPrompt

In [3]:
SEED = 0

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook4-sft-trainer-llama-1b"

TRAIN_SAMPLES = None
EVAL_SAMPLES = None
EVAL_GENERATION_SAMPLES = 32

MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512

LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
TRAIN_BATCH_SIZE = 16 ## Try smaller batch size on colab but greater accumulation size
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
LOGGING_STEPS = 10
SAVE_STEPS = 50


torch.manual_seed(SEED)

In [4]:
if torch.cuda.is_available():
    device = "cuda"
    model_dtype = "auto"
elif torch.backends.mps.is_available():
    device = "mps"
    model_dtype = torch.bfloat16
else:
    device = "cpu"
    model_dtype = torch.float32

print("Device:", device)
print("Model dtype:", model_dtype)

Device: cuda
Model dtype: auto


## Load the Tokenizer and Model

We fine-tune an **instruction-tuned** model.

That matters because the model expects a chat-formatted prompt,
not just a raw question string.


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=model_dtype,
).to(device=device)

model.config.use_cache = False
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

Tokenizer loaded.
Pad token: <|eot_id|>
EOS token: <|eot_id|>


## Load GSM8K

For training, we use a subset of the GSM8K train split.
For evaluation, we use a subset of the test split.

We keep the subsets small so the notebook stays practical.


The data can also be loaded from disk using the `Dataset.from_*` methods. Here, we will load the data from a JSONL file.

In [6]:
Dataset.from_*?

Dataset.from_buffer
Dataset.from_csv
Dataset.from_dict
Dataset.from_file
Dataset.from_generator
Dataset.from_json
Dataset.from_list
Dataset.from_pandas
Dataset.from_parquet
Dataset.from_polars
Dataset.from_spark
Dataset.from_sql
Dataset.from_text

In [7]:
# train_raw = load_dataset("openai/gsm8k", "main", split="train")
# eval_raw = load_dataset("openai/gsm8k", "main", split="test")

train_raw = Dataset.from_json("gsm8k/train.jsonl",encoding="utf-8")
eval_raw = Dataset.from_json("gsm8k/test.jsonl",encoding="utf-8")

In [8]:
if TRAIN_SAMPLES is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(TRAIN_SAMPLES))

if EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(EVAL_SAMPLES))

print(train_raw)
print(eval_raw)

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})
Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})


In [9]:
print(train_raw[0]["question"])
print()
print(train_raw[0]["answer"])

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## Format Each Example for an Instruct Model

Each training sample has two parts:

- the **prompt**: system + user messages
- the **completion**: the assistant answer we want the model to learn

Here we keep the GSM8K worked solution as the assistant target.
That means the model is trained to generate a reasoning trace followed by the final answer after `####`.


In [10]:
def format_example(example: dict) -> dict:
    prompt_messages = [
        {"role": "system", "content": "You are an expert in Math."},
        {"role": "user", "content": MATH_PROMPT.format(question=example["question"])},
    ]

    completion_text = example["answer"].strip()
    y_true = extract_last_after_hashes(example["answer"])

    ## LLMs usually trained to answer math questions in \boxed{...}
    completion_text = replace_hash_with_boxed(completion_text)
    

    full_messages = [
        *prompt_messages,
        {"role": "assistant", "content": completion_text},
    ]

    return {
        "prompt_messages": prompt_messages,
        "full_messages": full_messages,
        "completion_text": completion_text,
        "y_true": y_true.strip().lower() if y_true is not None else None,
    }


def tokenize(examples: dict) -> dict:
    prompt_ids = tokenizer.apply_chat_template(
        examples["prompt_messages"],
        tokenize=True,
        add_generation_prompt=True,
    )

    input_ids = tokenizer.apply_chat_template(
        examples["full_messages"],
        tokenize=True,
        add_generation_prompt=False,
    )

    return {"prompt_ids": prompt_ids, "input_ids": input_ids}


train_formatted = train_raw.map(format_example).map(tokenize, batched=True)
eval_formatted = eval_raw.map(format_example).map(tokenize, batched=True)

train_formatted

Dataset({
    features: ['question', 'answer', 'prompt_messages', 'full_messages', 'completion_text', 'y_true', 'prompt_ids', 'input_ids'],
    num_rows: 7473
})

In [11]:
print(tokenizer.decode(train_formatted[0]["prompt_ids"]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Apr 2026

You are an expert in Math.<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math word problem carefully.

Think step by step to ensure the reasoning is correct.
When you are confident in the final answer, output ONLY:

\boxed{FINAL_ANSWER}

Do not include any additional text in the box.

Problem:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [12]:
print(tokenizer.decode(train_formatted[0]["input_ids"]))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Apr 2026

You are an expert in Math.<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math word problem carefully.

Think step by step to ensure the reasoning is correct.
When you are confident in the final answer, output ONLY:

\boxed{FINAL_ANSWER}

Do not include any additional text in the box.

Problem:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
\boxed{72}<|eot_id|>


## Why We Must Mask the Prompt

For causal LM training, the model predicts the next token at every position.

If we concatenate:
- prompt tokens
- assistant completion tokens

then the raw training sequence is valid.
But the loss should not be computed everywhere.

For SFT on an instruct model, we want:
- the prompt to be **conditioning context**
- the assistant completion to be the **supervised target**

So we build:
- `input_ids = prompt_ids + completion_ids + [eos]`
- `labels = [-100] * len(prompt_ids) + completion_ids + [eos]`

In PyTorch cross-entropy, label `-100` means:
"ignore this position when computing the loss."

Without masking, the model would also be trained to predict:
- system tokens
- user tokens
- chat-template markers

That is not the learning objective we want.


In [13]:
def prepare_sft_samples(example: dict) -> dict:
    labels = example["input_ids"].copy()
    labels[:len(example["prompt_ids"])] = [-100] * len(example["prompt_ids"])

    input_ids = example["input_ids"][:MAX_LENGTH]
    labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "labels": labels
    }


example_tokens = prepare_sft_samples(train_formatted[0])


In [14]:
def show_masked_example(tokenizer, tokenized_example, max_rows: int = 400):
    print(f"{'idx':>4} | {'token':<18} | {'label':<8} | role")
    print("-" * 60)

    input_ids = tokenized_example["input_ids"]
    labels = tokenized_example["labels"]

    for i, (token_id, label) in enumerate(zip(input_ids[:max_rows], labels[:max_rows])):
        token = tokenizer.decode([token_id]).replace("\n", "\\n")
        region = "ignored" if label == -100 else "loss"
        print(f"{i:>4} | {token[:18]:<18} | {str(label):<8} | {region}")


show_masked_example(tokenizer, example_tokens)

 idx | token              | label    | role
------------------------------------------------------------
   0 | <|begin_of_text|>  | -100     | ignored
   1 | <|start_header_id| | -100     | ignored
   2 | system             | -100     | ignored
   3 | <|end_header_id|>  | -100     | ignored
   4 | \n\n               | -100     | ignored
   5 | Cut                | -100     | ignored
   6 | ting               | -100     | ignored
   7 |  Knowledge         | -100     | ignored
   8 |  Date              | -100     | ignored
   9 | :                  | -100     | ignored
  10 |  December          | -100     | ignored
  11 |                    | -100     | ignored
  12 | 202                | -100     | ignored
  13 | 3                  | -100     | ignored
  14 | \n                 | -100     | ignored
  15 | Today              | -100     | ignored
  16 |  Date              | -100     | ignored
  17 | :                  | -100     | ignored
  18 |                    | -100     | ignored
  

In the table above:
- the prompt region has label `-100`
- the assistant completion region has normal token IDs

That is the core SFT setup for instruction tuning.


## Build the Training and Evaluation Datasets

We map the tokenization function over the formatted datasets and keep only the columns needed for training or later evaluation.


In [15]:
train_tokenized = train_formatted.map(prepare_sft_samples)
eval_tokenized = eval_formatted.map(prepare_sft_samples)

keep_columns = {
    "input_ids",
    "prompt_ids",
    "attention_mask",
    "labels",
    "y_true",
}

train_dataset = train_tokenized.remove_columns(
    [c for c in train_tokenized.column_names if c not in keep_columns]
)
eval_dataset = eval_tokenized.remove_columns(
    [c for c in eval_tokenized.column_names if c not in keep_columns]
)


train_dataset

Dataset({
    features: ['y_true', 'prompt_ids', 'input_ids', 'labels'],
    num_rows: 7473
})

In [16]:
def build_attention_mask_from_lengths(
    lengths: list[int], max_len: int = None
) -> torch.Tensor:
    max_len = max(lengths) if max_len is None else max_len
    lengths = torch.tensor(lengths, dtype=torch.long)
    mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)
    return mask.long()


class SFTDataCollator:
    pad_token_id: int
    label_pad_token_id: int = -100

    def __init__(self, pad_token_id: int, label_pad_token_id: int = -100):
        self.pad_token_id = pad_token_id
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, examples: list[dict]) -> dict[str, torch.Tensor]:
        input_ids = [torch.tensor(x["input_ids"], dtype=torch.long) for x in examples]
        labels = [torch.tensor(x["labels"], dtype=torch.long) for x in examples]

        input_lens = [len(x) for x in input_ids]

        input_ids = pad_sequence(
            input_ids, batch_first=True, padding_value=self.pad_token_id
        )
        labels = pad_sequence(
            labels, batch_first=True, padding_value=self.label_pad_token_id
        )

        ## If the eos_token is not equal to pad_token
        # attention_mask = (input_ids != self.pad_token_id).long()
        attention_mask = build_attention_mask_from_lengths(input_lens)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


data_collator = SFTDataCollator(pad_token_id=tokenizer.pad_token_id)

In [17]:
lens = [len(x["input_ids"]) for x in train_dataset]

In [18]:
max(lens)

554

## A Small Evaluation Helper

We evaluate by generation, not by training loss alone.

That is important because SFT minimizes token-level cross-entropy,
but we ultimately care about the generated final answer.


In [19]:
baseline_subset, baseline_responses, baseline_metrics = evaluate_model(
    model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=3,
    batch_size=EVAL_BATCH_SIZE,
)

print(f"Baseline accuracy on {baseline_metrics['total']} samples: {baseline_metrics['accuracy']:.4f}")

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Baseline accuracy on 3 samples: 0.0000


In [20]:
baseline_responses

['16 * 3 = 48\n48 - 4 = 44\n44 / 2 = 22\n\nFINAL_ANSWER = 22',
 '2 * 0.5 = 1\n\nFINAL_ANSWER',
 'To find the profit, first calculate the increased value of the house:\n\n$80,000 + ($50,000 x 1.5) = $80,000 + $75,000 = $155,000\n\nThen, subtract the original purchase price from the increased value:\n\n$155,000 - $80,000 = $75,000\n\nSo, Josh made a profit of $75,000.']

In [21]:
print(tokenizer.decode(baseline_subset[0]["prompt_ids"]))
print()
Markdown(baseline_responses[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 12 Apr 2026

You are an expert in Math.<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math word problem carefully.

Think step by step to ensure the reasoning is correct.
When you are confident in the final answer, output ONLY:

\boxed{FINAL_ANSWER}

Do not include any additional text in the box.

Problem:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|eot_id|><|start_header_id|>assistant<|end_header_id|>





16 * 3 = 48
48 - 4 = 44
44 / 2 = 22

FINAL_ANSWER = 22

## Configure `Trainer`

`Trainer` is HuggingFace's high-level training loop for PyTorch models.

Why it is important:
- it handles the optimization loop, logging, evaluation, and checkpoint saving
- it integrates directly with tokenized datasets and custom collators
- it is the standard API that later extends naturally to PEFT, Accelerate, and DeepSpeed

A few settings matter here:

- `remove_unused_columns=False` so our custom collator receives the expected fields
- `gradient_checkpointing=True` to reduce activation memory
- `bf16` or `fp16` depending on the hardware
- `lr_scheduler_type="cosine"` to show that the schedule is configurable

We also disable cache during training because caching is useful for generation, not for gradient-based optimization.


## Checkpointing

In this notebook, checkpointing is used for **resumption**.

`Trainer` saves model weights together with optimizer, scheduler, and trainer state so that training can continue later from the last saved checkpoint.

This is different from **best-model selection**. We are not asking `Trainer` to monitor validation performance and automatically reload the best checkpoint.

If you want that behavior later, you would typically enable:
- `load_best_model_at_end=True`
- `metric_for_best_model="eval_loss"`
- `greater_is_better=False`

For now, we keep checkpointing simple and use it only for saving progress and resuming interrupted training.


In [22]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,  # Directory where checkpoints and logs are written.
    overwrite_output_dir=True,  # Reuse the same output directory across notebook runs.
    num_train_epochs=NUM_EPOCHS,  # Number of full passes over the training set.
    learning_rate=LEARNING_RATE,  # Peak learning rate for AdamW.
    lr_scheduler_type="cosine",  # Decay the learning rate with a cosine schedule.
    warmup_ratio=WARMUP_RATIO,  # Fraction of training reserved for warmup.
    weight_decay=WEIGHT_DECAY,  # L2-style regularization on weights.
    per_device_train_batch_size=TRAIN_BATCH_SIZE,  # Micro-batch size per device for training.
    per_device_eval_batch_size=EVAL_BATCH_SIZE,  # Micro-batch size per device for evaluation.
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,  # Simulate a larger effective batch size.
    # eval_strategy="epoch",  # Evaluate once per epoch.
    save_strategy="steps",  # Save checkpoints every fixed number of optimizer steps.
    save_steps=SAVE_STEPS,  # Checkpoint interval when using step-based saving.
    # save_strategy="epoch",  # Alternative: save one checkpoint at the end of each epoch.
    save_total_limit=2,  # Keep only the two most recent checkpoints.
    # load_best_model_at_end=True,  # Reload the best validation checkpoint at the end.
    # metric_for_best_model="eval_loss",  # Use validation loss to choose the best checkpoint.
    # greater_is_better=False,  # Lower eval loss means a better checkpoint.
    logging_strategy="steps",  # Log every fixed number of optimizer steps.
    logging_steps=LOGGING_STEPS,  # Step interval used for logging.
    remove_unused_columns=False,  # Keep custom dataset fields for the collator.
    gradient_checkpointing=True,  # Reduce activation memory during training.
    bf16=(
        device != "cpu" and model_dtype == torch.bfloat16
    ),  # Enable bf16 mixed precision when appropriate.
    fp16=(
        device != "cpu" and model_dtype == torch.float16
    ),  # Enable fp16 mixed precision when appropriate.
    report_to="tensorboard",  # "none" disables external experiment trackers for this notebook.
    dataloader_num_workers=2,  # Use worker processes for faster batch preparation.
)

trainer = Trainer(
    model=model,  # The causal language model being fine-tuned.
    args=training_args,  # All training, logging, evaluation, and checkpoint settings.
    train_dataset=train_dataset,  # Tokenized training dataset with masked labels.
    # eval_dataset=eval_dataset,  # Tokenized validation dataset.
    data_collator=data_collator,  # Pads variable-length examples into dense tensors.
    tokenizer=tokenizer,  # Saved alongside checkpoints for reproducibility.
)

/tmp/ipykernel_398241/797351696.py:34: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
[2026-04-12 18:48:39] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -c /tmp/tmpzbg7p2f_/test.c -o /tmp/tmpzbg7p2f_/test.o


[2026-04-12 18:48:39] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmpzbg7p2f_/test.o -laio -o /tmp/tmpzbg7p2f_/a.out
/nfs/home/seymol/conda_envs/v3.13/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
[2026-04-12 18:48:39] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DNDEBUG -O2 -Wall -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -O2 -isystem /nfs/home/seymol/conda_envs/v3.13/include -fPIC -c /tmp/tmpz80rx1jt/test.c -o /tmp/tmpz80rx1jt/test.o
[2026-04-12 18:48:39] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat /tmp/tmpz80rx1jt/test.o -L/usr/local/cuda -L/usr/local/cuda/lib64 -lcufile -o /tmp/tmpz80rx1jt/a.out
[2026-04-12 18:48:39] INFO spawn.py:77: gcc -pthread -B /nfs/home/seymol/conda_envs/v3.13/compiler_compat -fno-strict-overflow -Wsign-compare -DND

## Train

This is ordinary supervised fine-tuning.

At each step, the loss is computed only on the unmasked assistant tokens.


In [23]:
checkpoint = None
if os.path.isdir(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)

print("Resuming from checkpoint:", checkpoint)
train_result = trainer.train(resume_from_checkpoint=checkpoint)
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Resuming from checkpoint: None


Step,Training Loss
10,0.927200
20,0.727000
30,0.616700
40,0.537600
50,0.538200
60,0.528200
70,0.513300
80,0.508700
90,0.495900
100,0.495500


TrainOutput(global_step=1170, training_loss=0.33603646816351473, metrics={'train_runtime': 1451.6765, 'train_samples_per_second': 25.739, 'train_steps_per_second': 0.806, 'total_flos': 8.200886794326835e+16, 'train_loss': 0.33603646816351473, 'epoch': 5.0})

## Evaluate After Fine-Tuning

Now we run the same generation-based evaluation again.


In [24]:
ft_subset, ft_responses, ft_metrics = evaluate_model(
    model,
    tokenizer,
    eval_dataset,
    max_new_tokens=MAX_NEW_TOKENS,
    sample_count=EVAL_GENERATION_SAMPLES,
    batch_size=EVAL_BATCH_SIZE,
)

print(f"Fine-tuned accuracy on {ft_metrics['total']} samples: {ft_metrics['accuracy']:.4f}")

Fine-tuned accuracy on 32 samples: 0.2188


In [25]:
print("Baseline accuracy:", round(baseline_metrics["accuracy"], 4))
print("Fine-tuned accuracy:", round(ft_metrics["accuracy"], 4))

Baseline accuracy: 0.0
Fine-tuned accuracy: 0.2188


In [26]:
Markdown(ft_responses[0])

She has 16-3 = <<16-3=13>>13 eggs left every day.
She bakes muffins for 13-4 = <<13-4=9>>9 days.
She makes 9*2 = $<<9*2=18>>18 every day from the muffins.
She makes 13*2 = $<<13*2=26>>26 every day from the eggs.
She makes 18+26 = $<<18+26=44>>44 every day at the farmers' market.
\boxed{44}

## Save and Load the Fine-Tuned Model

We save both the model and tokenizer so the checkpoint can be reused later.

Loading is symmetric:
- load the tokenizer from the checkpoint directory
- load the model from the same directory
- switch the model to evaluation mode before inference
- re-enable caching for fast autoregressive generation


In [27]:
MODEL_DIR = os.path.join(OUTPUT_DIR,"final_model")

In [28]:
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print("Saved to:", MODEL_DIR)

Saved to: artifacts/notebook4-sft-trainer-llama-1b/final_model


In [29]:
reloaded_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
reloaded_model = AutoModelForCausalLM.from_pretrained(MODEL_DIR).to(device)
reloaded_model.eval()
reloaded_model.config.use_cache = True

print("Reloaded checkpoint from:", MODEL_DIR)

The tokenizer you are loading from 'artifacts/notebook4-sft-trainer-llama-1b/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Reloaded checkpoint from: artifacts/notebook4-sft-trainer-llama-1b/final_model


## vLLM Evaluation (After Cleaning CUDA Memory)

We clear the GPU memory used by the PyTorch model before initializing vLLM.
Then we run the same GSM8K-style evaluation using the vLLM runtime.


### Important: Restart the Kernel Before vLLM

PyTorch can keep GPU memory fragments alive even after `del` and `empty_cache()`.
For vLLM evaluation, **restart the kernel** and run only
the minimal setup cells below.


In [1]:
# After restart: re-import minimal dependencies
import os

import torch
from transformers import AutoTokenizer

from utils import build_gsm8k_sft_datasets, compute_accuracy

from vllm import LLM, SamplingParams, TokensPrompt

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # HuggingFaceTB/SmolLM2-135M-Instruct
OUTPUT_DIR = "artifacts/notebook4-sft-trainer-llama-1b"

MODEL_DIR = os.path.join(OUTPUT_DIR,"final_model")

SEED = 0
MAX_LENGTH = 1024
MAX_NEW_TOKENS = 512

train_dataset, eval_dataset = build_gsm8k_sft_datasets(
    tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME),
    train_path="gsm8k/train.jsonl",
    eval_path="gsm8k/test.jsonl",
    seed=SEED,
    max_length=MAX_LENGTH,
)

In [4]:
# Build prompts from the tokenized dataset
prompt_tokens = [TokensPrompt(prompt_token_ids=x) for x in eval_dataset["prompt_ids"]]

llm = LLM(
    model=MODEL_DIR,
    max_model_len=MAX_LENGTH + MAX_NEW_TOKENS,
    enforce_eager=True,
    gpu_memory_utilization=0.9,
)

sampling_params = SamplingParams(
    max_tokens=MAX_NEW_TOKENS,
    temperature=0.0,
)

outputs = llm.generate(prompt_tokens, sampling_params)
vllm_responses = [out.outputs[0].text for out in outputs]

vllm_metrics = compute_accuracy(responses=vllm_responses, dataset=eval_dataset)
print(f"vLLM accuracy on {vllm_metrics['total']} samples: {vllm_metrics['accuracy']:.4f}")


INFO 04-12 19:39:44 [utils.py:253] non-default args: {'seed': None, 'max_model_len': 1536, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'artifacts/notebook4-sft-trainer-llama-1b/final_model'}
WARNING 04-12 19:39:44 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.


INFO 04-12 19:39:44 [model.py:637] Resolved architecture: LlamaForCausalLM
INFO 04-12 19:39:44 [model.py:1750] Using max model len 1536
INFO 04-12 19:39:44 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 04-12 19:39:44 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 04-12 19:39:44 [vllm.py:707] Cudagraph is disabled under eager mode


The tokenizer you are loading from 'artifacts/notebook4-sft-trainer-llama-1b/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=539429) INFO 04-12 19:39:45 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='artifacts/notebook4-sft-trainer-llama-1b/final_model', speculative_config=None, tokenizer='artifacts/notebook4-sft-trainer-llama-1b/final_model', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1536, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_tr

(EngineCore_DP0 pid=539429) /nfs/home/seymol/conda_envs/v3.13/lib/python3.13/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:181: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=539429) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=539429)   warnings.warn(


...

(EngineCore_DP0 pid=539429) The tokenizer you are loading from 'artifacts/notebook4-sft-trainer-llama-1b/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


(EngineCore_DP0 pid=539429) WARNING 04-12 19:40:02 [vllm.py:608] Inductor compilation was disabled by user settings,Optimizations settings that are only active duringInductor compilation will be ignored.
(EngineCore_DP0 pid=539429) INFO 04-12 19:40:02 [vllm.py:707] Cudagraph is disabled under eager mode
INFO 04-12 19:40:02 [llm.py:343] Supported tasks: ['generate']


Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

vLLM accuracy on 1319 samples: 0.3973


In [5]:
print(vllm_responses[0])

She has 16-3 = <<16-3=13>>13 eggs left every day.
She bakes muffins for 13-4 = <<13-4=9>>9 days.
She makes 9*2 = $<<9*2=18>>18 every day from the muffins.
She makes 13*2 = $<<13*2=26>>26 every day from the eggs.
She makes 18+26 = $<<18+26=44>>44 every day at the farmers' market.
\boxed{44}


## Summary

In this notebook we built the standard SFT pipeline for an instruct model:

- format the prompt with the chat template
- append the gold assistant completion
- mask prompt tokens with `-100`
- fine-tune with `Trainer`
- evaluate by generation

The most important lesson is the masking rule:

**the prompt is context, the assistant answer is the supervised target.**

In the next notebook, we will keep the same training idea,
but make it much more memory efficient with PEFT / LoRA.
